# 03 - Gemini Connector Exploration

**Goal**: Explore the Gemini public REST API to inform our production connector.

**Scope**:
- Test raw `httpx` approach (no existing SDK — custom connector per DEC-005/DEC-011)
- Map Gemini symbols to canonical pairs
- Parse responses into our `TopOfBook` dataclass
- Document rate limits, error handling, edge cases
- Answer key design questions for the production connector

**Supported Pairs** (from PROJECT_INSTRUCTIONS.md):
- BTC/USD, BTC/USDC
- LTC/USD, LTC/USDC, LTC/BTC
- SOL/USD, SOL/USDC, SOL/BTC

**Key Differences from Kraken & Coinbase**:
- Gemini uses lowercase, no-separator symbols: `btcusd` (not `XXBTZUSD` or `BTC-USD`)
- No existing Python SDK for async use — raw httpx from the start
- Public endpoints: 120 req/min, recommended ≤1 req/sec
- Ticker endpoints (V1 & V2) have bid/ask prices but **NO bid/ask sizes**
- Must use order book endpoint (`/v1/book/{symbol}`) for TopOfBook with sizes
- Order book timestamps are Unix seconds (strings), not ISO 8601

**Lessons Applied** (from LESSONS_LEARNED.md):
- LL-001: Verify exact symbol format, don't assume
- LL-002: Document actual response shapes from live API, not just docs
- LL-003: Test rate limit behavior before building production connector
- LL-010: All prices/sizes via `to_decimal()`, never float
- LL-050: Use `nest_asyncio.apply()` for async in Jupyter
- LL-052: No batch endpoint assumption — verify before building connector

## 1. Setup

In [1]:
# Install dependencies (run once)
# !pip install httpx nest_asyncio

In [2]:
import json
import sys
import time
from pprint import pprint

import httpx
import nest_asyncio

# Enable nested event loops for Jupyter (LL-050)
nest_asyncio.apply()

sys.path.insert(0, "../src")

# Our existing infrastructure — reuse, don't reimplement
from uscryptoarb.marketdata.topofbook import TopOfBook, tob_from_raw
from uscryptoarb.validation.guards import require_present
from uscryptoarb.venues.symbols import SymbolTranslator

BASE_URL = "https://api.gemini.com"

print("Setup complete.")

Setup complete.


## 2. Symbol Discovery & Mapping

Gemini uses lowercase, no-separator symbols: `btcusd`, `ltcbtc`, `solusdc`.
Let's verify all 8 target pairs exist and build the symbol map.

In [3]:
# Fetch all available symbols
resp = httpx.get(f"{BASE_URL}/v1/symbols")
all_symbols = resp.json()
print(f"Total symbols on Gemini: {len(all_symbols)}")
print(f"\nFirst 20: {all_symbols[:20]}")

Total symbols on Gemini: 447

First 20: ['2zgusd', '2zrlusd', '2zusd', '2zusdc', 'aavegusd', 'aaverlusd', 'aaveusd', 'aaveusdc', 'aligusd', 'alirlusd', 'aliusd', 'aliusdc', 'ampgusd', 'amprlusd', 'ampusd', 'ampusdc', 'ankrgusd', 'ankrrlusd', 'ankrusd', 'ankrusdc']


In [4]:
# Our canonical pairs → Gemini symbols
# Format: lowercase, no separator (BTC/USD → btcusd)
GEMINI_SYMBOL_MAP = {
    "BTC/USD": "btcusd",
    "BTC/USDC": "btcusdc",
    "LTC/USD": "ltcusd",
    "LTC/USDC": "ltcusdc",
    "LTC/BTC": "ltcbtc",
    "SOL/USD": "solusd",
    "SOL/USDC": "solusdc",
    "SOL/BTC": "solbtc",
}

# Verify all 8 target pairs exist on Gemini
found_pairs = {}
missing_pairs = []

for canonical, gemini_sym in GEMINI_SYMBOL_MAP.items():
    if gemini_sym in all_symbols:
        found_pairs[canonical] = gemini_sym
        print(f"  ✅ {canonical} → {gemini_sym}")
    else:
        missing_pairs.append(canonical)
        print(f"  ❌ {canonical} → {gemini_sym} NOT FOUND")

print(f"\nFound: {len(found_pairs)}/8 | Missing: {missing_pairs or 'None'}")

  ✅ BTC/USD → btcusd
  ✅ BTC/USDC → btcusdc
  ✅ LTC/USD → ltcusd
  ✅ LTC/USDC → ltcusdc
  ✅ LTC/BTC → ltcbtc
  ✅ SOL/USD → solusd
  ✅ SOL/USDC → solusdc
  ✅ SOL/BTC → solbtc

Found: 8/8 | Missing: None


In [5]:
# Build SymbolTranslator (same pattern as Kraken/Coinbase)
# Requires: venue name + canonical→venue mapping dict
gemini_translator = SymbolTranslator(venue="gemini", canonical_to_venue=GEMINI_SYMBOL_MAP)

# Test round-trip
for canonical in GEMINI_SYMBOL_MAP:
    exchange_sym = gemini_translator.to_venue_symbol(canonical)
    back = gemini_translator.to_canonical(exchange_sym)
    assert back == canonical, f"Round-trip failed: {canonical} → {exchange_sym} → {back}"

print("SymbolTranslator round-trip: all 8 pairs pass ✅")

SymbolTranslator round-trip: all 8 pairs pass ✅


In [6]:
# DEC-001 verification: USD ≠ USDC
# Verify that btcusd and btcusdc are distinct symbols with distinct quote currencies
for sym in ["btcusd", "btcusdc"]:
    details = httpx.get(f"{BASE_URL}/v1/symbols/details/{sym}").json()
    print(
        f"{sym}: base={details['base_currency']}, quote={details['quote_currency']}, "
        f"status={details['status']}"
    )

print("\nUSD ≠ USDC confirmed: distinct quote currencies ✅")

btcusd: base=BTC, quote=USD, status=open
btcusdc: base=BTC, quote=USDC, status=open

USD ≠ USDC confirmed: distinct quote currencies ✅


## 3. Ticker Endpoints (V1 & V2)

Gemini has two ticker versions. Let's explore both to understand what data is available.

**Key question**: Do either provide bid/ask sizes? (Spoiler from docs: no.)

In [7]:
# Ticker V1: GET /v1/pubticker/{symbol}
ticker_v1 = httpx.get(f"{BASE_URL}/v1/pubticker/btcusd").json()
print("Ticker V1 (btcusd):")
print(json.dumps(ticker_v1, indent=2))
print(f"\nV1 keys: {sorted(ticker_v1.keys())}")
print(f"bid type: {type(ticker_v1['bid']).__name__} = {ticker_v1['bid']}")
print(f"ask type: {type(ticker_v1['ask']).__name__} = {ticker_v1['ask']}")
print("\n⚠️  No bid_size or ask_size in V1 response!")

Ticker V1 (btcusd):
{
  "bid": "68290.96",
  "ask": "68290.97",
  "last": "68260.86",
  "volume": {
    "BTC": "224.77200433",
    "USD": "15343130.3194895238",
    "timestamp": 1771210763000
  }
}

V1 keys: ['ask', 'bid', 'last', 'volume']
bid type: str = 68290.96
ask type: str = 68290.97

⚠️  No bid_size or ask_size in V1 response!


In [8]:
# Ticker V2: GET /v2/ticker/{symbol}
ticker_v2 = httpx.get(f"{BASE_URL}/v2/ticker/btcusd").json()
print("Ticker V2 (btcusd):")
print(json.dumps(ticker_v2, indent=2))
print(f"\nV2 keys: {sorted(ticker_v2.keys())}")
print("\n⚠️  V2 also has NO bid/ask sizes — only OHLC + bid/ask prices + hourly changes")

Ticker V2 (btcusd):
{
  "symbol": "BTCUSD",
  "open": "68809.87",
  "high": "70938.52",
  "low": "68047.23",
  "close": "69482.33",
  "changes": [
    "68587.86",
    "68809.87",
    "68786.33",
    "68919.2",
    "68834.63",
    "68414.94",
    "68301.11",
    "68393.83",
    "68596.59",
    "69024.71",
    "69058.65",
    "69112.82",
    "68962.2",
    "69297.73",
    "70267.23",
    "70404.81",
    "70386.19",
    "70401.08",
    "70738.92",
    "70367.31",
    "70235.14",
    "70133.63",
    "69947.39",
    "69482.33"
  ],
  "bid": "68290.96",
  "ask": "68290.97"
}

V2 keys: ['ask', 'bid', 'changes', 'close', 'high', 'low', 'open', 'symbol']

⚠️  V2 also has NO bid/ask sizes — only OHLC + bid/ask prices + hourly changes


In [9]:
# Price Feed: GET /v1/pricefeed — returns ALL pairs (potential batch alternative?)
pricefeed = httpx.get(f"{BASE_URL}/v1/pricefeed").json()
print(f"Price feed has {len(pricefeed)} entries")

# Filter to our target pairs
target_symbols_upper = {v.upper() for v in GEMINI_SYMBOL_MAP.values()}
our_pairs = [p for p in pricefeed if p["pair"] in target_symbols_upper]
print(f"\nOur pairs in pricefeed ({len(our_pairs)}/8):")
for p in our_pairs:
    print(f"  {p['pair']}: price={p['price']}, change24h={p['percentChange24h']}")

print("\n⚠️  Pricefeed only has last price — no bid/ask/size. Not usable for TopOfBook.")

Price feed has 444 entries

Our pairs in pricefeed (8/8):
  BTCUSDC: price=68260.86, change24h=-0.0142
  LTCBTC: price=0.0007957, change24h=-0.0142
  BTCUSD: price=68260.86, change24h=-0.0142
  LTCUSD: price=54.5, change24h=-0.0189
  SOLUSD: price=85.292, change24h=-0.0217
  SOLUSDC: price=85.292, change24h=-0.0217
  LTCUSDC: price=54.5, change24h=-0.0189
  SOLBTC: price=0.0012524, change24h=-0.0152

⚠️  Pricefeed only has last price — no bid/ask/size. Not usable for TopOfBook.


### Ticker Conclusion

**Neither ticker V1 nor V2 provides bid/ask sizes.** The price feed is also only last-trade prices.

For `TopOfBook` data (which requires bid_px, bid_sz, ask_px, ask_sz), we **must** use the
order book endpoint. This is different from Kraken (whose ticker includes lot volumes)
but similar to Coinbase (which required `/market/product_book`).

## 4. Order Book Endpoint (Primary Data Source)

Since tickers lack sizes, the order book is our primary endpoint:
`GET /v1/book/{symbol}?limit_bids=1&limit_asks=1`

This returns top-of-book with prices, amounts, and timestamps.

In [10]:
# Fetch full order book (default: 50 levels each side)
book_full = httpx.get(f"{BASE_URL}/v1/book/btcusd").json()
print("Full order book (btcusd):")
print(f"  Bids: {len(book_full['bids'])} levels")
print(f"  Asks: {len(book_full['asks'])} levels")
print(f"\nTop bid: {json.dumps(book_full['bids'][0], indent=2)}")
print(f"Top ask: {json.dumps(book_full['asks'][0], indent=2)}")

# Document the response structure
print("\n--- Response Structure ---")
print("Each level: {price: str, amount: str, timestamp: str}")
sample = book_full["bids"][0]
print(f"  price type: {type(sample['price']).__name__} = {sample['price']}")
print(f"  amount type: {type(sample['amount']).__name__} = {sample['amount']}")
print(f"  timestamp type: {type(sample['timestamp']).__name__} = {sample['timestamp']}")
print("\n  ⚠️  Timestamp is Unix SECONDS as a string (not ms, not ISO 8601)")

Full order book (btcusd):
  Bids: 50 levels
  Asks: 50 levels

Top bid: {
  "price": "68290.96",
  "amount": "0.22861801",
  "timestamp": "1771210764"
}
Top ask: {
  "price": "68290.97",
  "amount": "0.01063328",
  "timestamp": "1771210764"
}

--- Response Structure ---
Each level: {price: str, amount: str, timestamp: str}
  price type: str = 68290.96
  amount type: str = 0.22861801
  timestamp type: str = 1771210764

  ⚠️  Timestamp is Unix SECONDS as a string (not ms, not ISO 8601)


In [11]:
# Fetch top-of-book only (limit_bids=1, limit_asks=1) — this is what we'll use
book_top = httpx.get(
    f"{BASE_URL}/v1/book/btcusd",
    params={"limit_bids": 1, "limit_asks": 1},
).json()

print("Top-of-book (btcusd):")
print(json.dumps(book_top, indent=2))

# Compare sizes — top-only should be much smaller response
print(f"\nFull book response size: ~{len(json.dumps(book_full))} bytes")
print(f"Top-only response size:  ~{len(json.dumps(book_top))} bytes")
print(f"Reduction: {100 * (1 - len(json.dumps(book_top)) / len(json.dumps(book_full))):.0f}%")

Top-of-book (btcusd):
{
  "bids": [
    {
      "price": "68290.96",
      "amount": "0.22861801",
      "timestamp": "1771210764"
    }
  ],
  "asks": [
    {
      "price": "68290.97",
      "amount": "0.01063328",
      "timestamp": "1771210764"
    }
  ]
}

Full book response size: ~7084 bytes
Top-only response size:  ~168 bytes
Reduction: 98%


In [12]:
# Fetch top-of-book for ALL 8 pairs
print("Top-of-book for all 8 target pairs:\n")
all_books = {}

for canonical, gemini_sym in GEMINI_SYMBOL_MAP.items():
    resp = httpx.get(
        f"{BASE_URL}/v1/book/{gemini_sym}",
        params={"limit_bids": 1, "limit_asks": 1},
    )
    book = resp.json()
    all_books[canonical] = book

    if book.get("bids") and book.get("asks"):
        bid = book["bids"][0]
        ask = book["asks"][0]
        print(
            f"  {canonical:10s} → bid={bid['price']:>12s} ({bid['amount']:>12s}) "
            f"| ask={ask['price']:>12s} ({ask['amount']:>12s})"
        )
    else:
        print(f"  {canonical:10s} → ⚠️  Empty book: {book}")

    time.sleep(0.6)  # Respect ~1 req/sec recommendation

Top-of-book for all 8 target pairs:

  BTC/USD    → bid=    68290.96 (  0.22861801) | ask=    68290.97 (  0.01063328)
  BTC/USDC   → bid=    68290.96 (  0.22861801) | ask=    68290.97 (  0.01063328)
  LTC/USD    → bid=       54.44 (    36.71045) | ask=       54.47 (    39.88814)
  LTC/USDC   → bid=       54.45 (      0.6328) | ask=       54.47 (    39.88814)
  LTC/BTC    → bid=   0.0007968 (     7.95904) | ask=   0.0007977 (     7.95904)
  SOL/USD    → bid=      85.369 (      11.554) | ask=       85.37 (    2.371434)
  SOL/USDC   → bid=      85.353 (   56.004435) | ask=      85.354 (    0.008934)
  SOL/BTC    → bid=   0.0012494 (    1.348296) | ask=     0.00125 (    1.348296)


## 5. Parse into TopOfBook

Build a prototype parser that converts Gemini order book responses into our `TopOfBook` dataclass.

In [13]:
def parse_gemini_book(
    gemini_symbol: str,
    canonical_pair: str,
    book_data: dict,
    ts_local_ms: int,
) -> TopOfBook:
    """
    Parse Gemini order book response into TopOfBook.

    Args:
        gemini_symbol: Gemini symbol (e.g., 'btcusd')
        canonical_pair: Our canonical pair (e.g., 'BTC/USD')
        book_data: The order book dict with 'bids' and 'asks' arrays
        ts_local_ms: Local timestamp when data was received

    Returns:
        TopOfBook instance

    Raises:
        ValueError: If data is missing or invalid
    """
    bids = require_present(book_data.get("bids"), f"{gemini_symbol}.bids")
    asks = require_present(book_data.get("asks"), f"{gemini_symbol}.asks")

    if not bids or not asks:
        raise ValueError(f"Empty book for {gemini_symbol}: bids={len(bids)}, asks={len(asks)}")

    best_bid = bids[0]
    best_ask = asks[0]

    # Gemini book timestamps are Unix seconds as strings
    # Use the more recent of bid/ask timestamps
    ts_bid = int(best_bid["timestamp"])
    ts_ask = int(best_ask["timestamp"])
    ts_exchange_ms = max(ts_bid, ts_ask) * 1000  # Convert seconds → milliseconds

    return tob_from_raw(
        venue="gemini",
        pair=canonical_pair,
        ts_local_ms=ts_local_ms,
        ts_exchange_ms=ts_exchange_ms,
        bid_px=best_bid["price"],
        bid_sz=best_bid["amount"],
        ask_px=best_ask["price"],
        ask_sz=best_ask["amount"],
    )


print("Parser function defined ✅")

Parser function defined ✅


In [14]:
# Test parser on all 8 pairs
ts_now = int(time.time() * 1000)
print("Parsing all 8 pairs into TopOfBook:\n")

for canonical, book in all_books.items():
    try:
        tob = parse_gemini_book(
            gemini_symbol=GEMINI_SYMBOL_MAP[canonical],
            canonical_pair=canonical,
            book_data=book,
            ts_local_ms=ts_now,
        )
        print(
            f"  ✅ {tob.pair:10s}: bid={tob.bid_px} ({tob.bid_sz}), "
            f"ask={tob.ask_px} ({tob.ask_sz}), "
            f"ts_exchange={tob.ts_exchange_ms}"
        )
    except Exception as e:
        print(f"  ❌ {canonical}: {e}")

Parsing all 8 pairs into TopOfBook:

  ✅ BTC/USD   : bid=68290.96 (0.22861801), ask=68290.97 (0.01063328), ts_exchange=1771210765000
  ✅ BTC/USDC  : bid=68290.96 (0.22861801), ask=68290.97 (0.01063328), ts_exchange=1771210765000
  ✅ LTC/USD   : bid=54.44 (36.71045), ask=54.47 (39.88814), ts_exchange=1771210766000
  ✅ LTC/USDC  : bid=54.45 (0.6328), ask=54.47 (39.88814), ts_exchange=1771210767000
  ✅ LTC/BTC   : bid=0.0007968 (7.95904), ask=0.0007977 (7.95904), ts_exchange=1771210768000
  ✅ SOL/USD   : bid=85.369 (11.554), ask=85.37 (2.371434), ts_exchange=1771210768000
  ✅ SOL/USDC  : bid=85.353 (56.004435), ask=85.354 (0.008934), ts_exchange=1771210769000
  ✅ SOL/BTC   : bid=0.0012494 (1.348296), ask=0.00125 (1.348296), ts_exchange=1771210770000


## 6. Async httpx Pattern (Production Preview)

Since there's no SDK, the production connector will use `httpx.AsyncClient` directly
(same pattern as Coinbase, per DEC-011). Let's test the async flow.

In [15]:
async def fetch_all_books_async(
    pairs: dict[str, str],
    delay_ms: int = 600,
) -> dict[str, TopOfBook]:
    """
    Fetch top-of-book for all pairs using async httpx.

    Sequential with delay to respect rate limits (same pattern as Coinbase).
    """
    import asyncio

    results: dict[str, TopOfBook] = {}
    errors: dict[str, str] = {}

    async with httpx.AsyncClient(base_url=BASE_URL) as client:
        for canonical, gemini_sym in pairs.items():
            int(time.time() * 1000)
            try:
                resp = await client.get(
                    f"/v1/book/{gemini_sym}",
                    params={"limit_bids": 1, "limit_asks": 1},
                )
                resp.raise_for_status()
                book = resp.json()
                ts_after = int(time.time() * 1000)

                tob = parse_gemini_book(
                    gemini_symbol=gemini_sym,
                    canonical_pair=canonical,
                    book_data=book,
                    ts_local_ms=ts_after,
                )
                results[canonical] = tob
            except Exception as exc:
                errors[canonical] = str(exc)

            await asyncio.sleep(delay_ms / 1000)

    return results, errors


# Run async fetch
start = time.time()
tobs, errs = await fetch_all_books_async(GEMINI_SYMBOL_MAP, delay_ms=600)
elapsed = time.time() - start

print(f"Fetched {len(tobs)} pairs in {elapsed:.2f}s ({elapsed / len(tobs):.3f}s/pair)\n")
for pair, tob in tobs.items():
    print(f"  {pair:10s}: bid={tob.bid_px}, ask={tob.ask_px}")
if errs:
    print(f"\nErrors: {errs}")

TypeError: cannot create weak reference to 'NoneType' object

## 7. Rate Limit Testing

Gemini docs state: 120 req/min for public endpoints (2 req/sec).
Burst of 5 additional queued requests allowed. 429 when exceeded.

Let's test with rapid requests to measure actual behavior.

In [ ]:
# Rapid burst test — 15 requests with no delay
print("Burst test: 15 rapid requests to /v1/book/btcusd\n")

latencies = []
status_codes = []

for i in range(15):
    start = time.time()
    resp = httpx.get(
        f"{BASE_URL}/v1/book/btcusd",
        params={"limit_bids": 1, "limit_asks": 1},
    )
    elapsed_ms = (time.time() - start) * 1000
    latencies.append(elapsed_ms)
    status_codes.append(resp.status_code)
    print(f"  Request {i + 1:2d}: {resp.status_code} in {elapsed_ms:.0f}ms")

n_429 = status_codes.count(429)
avg_ms = sum(latencies) / len(latencies)
print(f"\n429 responses: {n_429}/15")
print(f"Avg latency: {avg_ms:.0f}ms")
if n_429 == 0:
    print("✅ No rate limiting hit with 15 rapid requests")
else:
    print(f"⚠️  Hit rate limit at request {status_codes.index(429) + 1}")

In [ ]:
# Sustained test — 8 requests with 500ms delay (simulates production polling)
print("Sustained test: 8 requests with 500ms delay (simulates 1 polling cycle)\n")

sustained_latencies = []
sustained_errors = 0
start_total = time.time()

for i, (canonical, gemini_sym) in enumerate(GEMINI_SYMBOL_MAP.items()):
    start = time.time()
    resp = httpx.get(
        f"{BASE_URL}/v1/book/{gemini_sym}",
        params={"limit_bids": 1, "limit_asks": 1},
    )
    elapsed_ms = (time.time() - start) * 1000
    sustained_latencies.append(elapsed_ms)

    if resp.status_code != 200:
        sustained_errors += 1
        print(f"  {canonical}: ⚠️  {resp.status_code} in {elapsed_ms:.0f}ms")
    else:
        print(f"  {canonical}: 200 in {elapsed_ms:.0f}ms")

    time.sleep(0.5)  # 500ms between requests

total_time = time.time() - start_total
print(f"\nTotal cycle time: {total_time:.2f}s")
print(f"Avg latency: {sum(sustained_latencies) / len(sustained_latencies):.0f}ms")
print(f"Errors: {sustained_errors}/8")

# Production connector rate limiter recommendation
print("\n--- Rate Limiter Recommendation ---")
print("120 req/min = 2 req/sec = 500ms between requests")
print("With 8 pairs at 500ms: ~4s per cycle (fits within 5s polling interval)")
print("Recommend: RateLimiter interval = 500ms (conservative, matches Kraken)")

## 8. Error Handling

Test various error scenarios to document the response format.

In [ ]:
# Error: invalid symbol (404 expected)
resp = httpx.get(f"{BASE_URL}/v1/book/INVALIDPAIR")
print("Invalid symbol:")
print(f"  Status: {resp.status_code}")
print(f"  Headers: Content-Type={resp.headers.get('content-type')}")
try:
    print(f"  Body: {json.dumps(resp.json(), indent=2)}")
except Exception:
    print(f"  Body (text): {resp.text[:200]}")

In [ ]:
# Error: bad query parameters
resp = httpx.get(
    f"{BASE_URL}/v1/book/btcusd",
    params={"limit_bids": -1},
)
print("Bad params (limit_bids=-1):")
print(f"  Status: {resp.status_code}")
try:
    print(f"  Body: {json.dumps(resp.json(), indent=2)}")
except Exception:
    print(f"  Body (text): {resp.text[:200]}")

In [ ]:
# Error: wrong endpoint path
resp = httpx.get(f"{BASE_URL}/v1/nonexistent")
print("Wrong endpoint:")
print(f"  Status: {resp.status_code}")
try:
    print(f"  Body: {json.dumps(resp.json(), indent=2)}")
except Exception:
    print(f"  Body (text): {resp.text[:200]}")

In [ ]:
# Error: invalid symbol on ticker
resp = httpx.get(f"{BASE_URL}/v1/pubticker/FAKEPAIR")
print("Ticker with invalid symbol:")
print(f"  Status: {resp.status_code}")
try:
    body = resp.json()
    print(f"  Body: {json.dumps(body, indent=2)}")
    print("\n--- Error Response Structure ---")
    print(f"  Keys: {sorted(body.keys()) if isinstance(body, dict) else 'N/A'}")
except Exception:
    print(f"  Body (text): {resp.text[:200]}")

## 9. Symbol Details (Precision & Min Sizes)

Fetch trading precision details for all 8 pairs — needed for order sizing in Phase 4
and useful context for the connector.

In [ ]:
# Fetch symbol details for all 8 pairs
print(
    f"{'Pair':10s} {'Status':8s} {'Min Size':>12s} {'Tick Size':>12s} "
    f"{'Quote Inc':>12s} {'Product':>8s}"
)
print("-" * 75)

all_details = {}
for canonical, gemini_sym in GEMINI_SYMBOL_MAP.items():
    resp = httpx.get(f"{BASE_URL}/v1/symbols/details/{gemini_sym}")
    details = resp.json()
    all_details[canonical] = details

    print(
        f"{canonical:10s} {details.get('status', '?'):8s} "
        f"{details.get('min_order_size', '?'):>12s} "
        f"{str(details.get('tick_size', '?')):>12s} "
        f"{str(details.get('quote_increment', '?')):>12s} "
        f"{details.get('product_type', '?'):>8s}"
    )

    time.sleep(0.6)

In [ ]:
# Inspect one full details response
print("Full details for btcusd:")
pprint(all_details.get("BTC/USD", {}))

print("\nFull details for solbtc:")
pprint(all_details.get("SOL/BTC", {}))

In [ ]:
# Compare USD vs USDC pairs — do they have the same precision?
print("USD vs USDC precision comparison:")
for base in ["BTC", "LTC", "SOL"]:
    usd = all_details.get(f"{base}/USD", {})
    usdc = all_details.get(f"{base}/USDC", {})
    print(
        f"\n  {base}/USD  — tick={usd.get('tick_size')}, "
        f"quote_inc={usd.get('quote_increment')}, "
        f"min_size={usd.get('min_order_size')}"
    )
    print(
        f"  {base}/USDC — tick={usdc.get('tick_size')}, "
        f"quote_inc={usdc.get('quote_increment')}, "
        f"min_size={usdc.get('min_order_size')}"
    )

## 10. Timestamp Format Deep Dive

Important for staleness detection. Gemini uses Unix seconds in order book
responses — need to verify precision and conversion.

In [ ]:
# Analyze timestamps from order book
from datetime import UTC, datetime

book = httpx.get(
    f"{BASE_URL}/v1/book/btcusd",
    params={"limit_bids": 5, "limit_asks": 5},
).json()

print("Timestamp analysis (btcusd top 5 levels):\n")
now_unix = int(time.time())

for side_name, levels in [("Bids", book["bids"]), ("Asks", book["asks"])]:
    print(f"{side_name}:")
    for level in levels:
        ts_str = level["timestamp"]
        ts_int = int(ts_str)
        age_sec = now_unix - ts_int
        dt = datetime.fromtimestamp(ts_int, tz=UTC)
        print(
            f"  price={level['price']:>12s}  ts={ts_str}  "
            f"({dt.strftime('%H:%M:%S')} UTC, {age_sec}s ago)"
        )

# Key finding: integer seconds precision (no sub-second)
print("\n--- Timestamp Format ---")
print("Format: Unix seconds as string (integer, no decimals)")
print("Precision: 1 second")
print("Conversion to ms: int(timestamp_str) * 1000")
print("\nCompare: Kraken orderbook = Unix seconds (float)")
print("         Coinbase product_book = ISO 8601 with microseconds")
print("         Gemini order book = Unix seconds (integer string)")

## 11. Summary & Connector Design Notes

### Key Findings

| Question | Answer |
|----------|--------|
| Batch ticker endpoint? | **No.** Neither ticker V1/V2 nor pricefeed provides bid/ask sizes. No batch order book endpoint. Must use per-pair `/v1/book/{symbol}` calls (like Coinbase). |
| Ticker response fields? | V1: bid, ask, last, volume (NO sizes). V2: OHLC + bid/ask + hourly changes (NO sizes). |
| Primary data endpoint? | **`/v1/book/{symbol}?limit_bids=1&limit_asks=1`** — only endpoint with bid/ask prices AND sizes. |
| Rate limits? | 120 req/min public (~2 req/sec). 429 on exceed. Burst of 5 queued requests. |
| All 8 pairs exist? | **Yes.** All 8 confirmed: btcusd, btcusdc, ltcusd, ltcusdc, ltcbtc, solusd, solusdc, solbtc. |
| Timestamp format? | Unix **seconds** as **string** (e.g., `"1547147541"`). Integer precision, no sub-second. |
| Error response format? | (Documented in Section 8 — check output cells above after running) |
| Symbol format? | Lowercase, no separator: `btcusd`, `solusdc`, `ltcbtc` |
| USD ≠ USDC? | **Confirmed.** Distinct symbols, distinct quote_currency in details. |

### Production Connector Design

**Pattern**: Identical to Coinbase connector — inherits `BaseAsyncConnector` (DEC-018).

```
connectors/gemini/
  __init__.py
  symbols.py    — GEMINI_SYMBOL_MAP + create_translator()
  parser.py     — parse_book_response() → TopOfBook
  client.py     — GeminiClient(BaseAsyncConnector) → fetch_tickers()
```

**Key decisions for production connector**:

1. **Endpoint**: `/v1/book/{symbol}?limit_bids=1&limit_asks=1` (not ticker)
2. **Rate limiter**: 500ms interval (conservative; 120 req/min = 500ms)
3. **Timestamp parsing**: `int(ts_str) * 1000` → milliseconds (simpler than Coinbase ISO 8601)
4. **No SDK needed**: raw httpx, same as Coinbase
5. **Per-pair requests**: Sequential with rate limiting (8 pairs × 500ms = ~4s, within 5s polling interval)
6. **Error handling**: Check HTTP status codes, parse JSON error body
7. **Inherits retry/backoff from BaseAsyncConnector** — no new retry logic needed

### Comparison: Gemini vs Kraken vs Coinbase

| Feature | Kraken | Coinbase | Gemini |
|---------|--------|----------|--------|
| Symbol format | `XXBTZUSD` | `BTC-USD` | `btcusd` |
| Batch endpoint | ✅ Ticker (all pairs in 1 call) | ❌ Per-pair only | ❌ Per-pair only |
| BBO endpoint | Ticker (has bid/ask + sizes) | `/market/product_book` | `/v1/book?limit=1` |
| Bid/ask sizes in ticker | ✅ (lot_volume) | N/A (uses book) | ❌ (no sizes in ticker) |
| Timestamp format | Unix seconds (float) | ISO 8601 microseconds | Unix seconds (int string) |
| Rate limit | ~1 req/sec (decays) | 10 req/sec (by IP) | 2 req/sec (120/min) |
| SDK | python-kraken-sdk | coinbase-advanced-py | None (custom httpx) |
| Rate limiter interval | 500ms | 100ms | 500ms |
| Exchange timestamps | Only in orderbook | In all responses | Only in orderbook |

### Refactor Candidates (per Coding Rule 10.8)

- **DummyRateLimiter**: Now 3 callers (Kraken, Coinbase, Gemini tests) → extract to `tests/helpers.py` (already exists per file structure)
- **Unix timestamp parsing**: Kraken uses `float(ts) * 1000`, Gemini uses `int(ts) * 1000` — similar but not identical. Consider shared helper if pattern emerges.
- **Per-pair fetch loop**: Coinbase and Gemini both loop per-pair with rate limiting. Pattern is in `BaseAsyncConnector` already.

### Lessons Learned (to add to docs/LESSONS_LEARNED.md)

- **LL-060** (pending): Gemini ticker endpoints lack bid/ask sizes — must use order book.
  Rule: Always verify the chosen endpoint provides ALL fields needed for TopOfBook before building the connector.
- **LL-061** (pending): Gemini timestamps are Unix seconds as strings, not numbers.
  Rule: Parse timestamps from order book as `int(str_value) * 1000` for consistency with ms convention.

In [ ]:
# Generate fixture data for tests — save raw responses
# (Run this cell after verifying all outputs above)
import json

# Fetch fresh data for fixture generation
fixture_pairs = {
    "btcusd": "BTC/USD",
    "ltcbtc": "LTC/BTC",
    "solbtc": "SOL/BTC",
}

print("Fixture responses for test data:\n")
for gemini_sym, canonical in fixture_pairs.items():
    resp = httpx.get(
        f"{BASE_URL}/v1/book/{gemini_sym}",
        params={"limit_bids": 1, "limit_asks": 1},
    )
    data = resp.json()
    print(f"--- {canonical} ({gemini_sym}) ---")
    print(json.dumps(data, indent=2))
    print()
    time.sleep(0.6)

print("\n💡 Copy these responses to tests/fixtures/ when building the production connector.")
print("   Naming convention: gemini_book_btc_usd.json, gemini_book_ltc_btc.json, etc.")